# Intelligent Resource Allocation in Cloud Computing and Storage System
### Machine Learning based Predictive Workload Scaling & Optimization

**Academic Project & Research Submission**

---
### Project Objectives:
1. **Cloud Workload Modeling**: Simulate multi-tenant cloud workload telemetry (CPU demand, RAM demand, Storage demand, Active concurrent tasks, and Network I/O).
2. **Exploratory Data Analysis (EDA)**: Analyze feature distributions, resource saturation thresholds, and inter-resource correlations.
3. **Predictive Model Training**: Train and compare multiple ML classifiers (`RandomForestClassifier`, `DecisionTreeClassifier`, `LogisticRegression`).
4. **Rigorous Evaluation**: Evaluate using Accuracy, Precision, Recall, Confusion Matrix, and Feature Importance metrics.
5. **Edge/Cloud Deployment**: Export trained model (`.joblib`) for direct integration into Flask real-time Telemetry Dashboard.

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')
print('All libraries successfully imported!')

## Step 2: Load and Inspect Cloud Workload Dataset

In [ ]:
# Load workload dataset
df = pd.read_csv('cloud_workload_dataset.csv')
print('Dataset Shape (Rows, Columns):', df.shape)
df.head()

## Step 3: Statistical Summary & Demand Tier Distribution

In [ ]:
print(df.describe())

# Class distribution: 0 = LOW, 1 = MEDIUM, 2 = HIGH
class_counts = df['demand_tier'].value_counts().sort_index()
labels = ['LOW (0)', 'MEDIUM (1)', 'HIGH (2)']
print('\nDemand Tier Class Breakdown:')
for lbl, cnt in zip(labels, class_counts):
    pct = (cnt / len(df)) * 100
    print(f'{lbl}: {cnt} instances ({pct:.1f}%)')

## Step 4: Exploratory Data Analysis (EDA Visualizations)

In [ ]:
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.hist(df['cpu_demand'], bins=20, color='dodgerblue', edgecolor='black', alpha=0.6, label='CPU')
plt.hist(df['ram_demand'], bins=20, color='mediumseagreen', edgecolor='black', alpha=0.6, label='RAM')
plt.hist(df['storage_demand'], bins=20, color='mediumpurple', edgecolor='black', alpha=0.6, label='Storage')
plt.title('Resource Utilization Distributions (%)')
plt.xlabel('Resource Demand (%)')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
tier_names = ['LOW Demand', 'MEDIUM Demand', 'HIGH Demand']
colors = ['#38bdf8', '#fbbf24', '#f87171']
plt.bar(tier_names, class_counts, color=colors, edgecolor='black')
plt.title('Distribution of Cloud Allocation Tiers')
plt.ylabel('Number of Requests')
plt.tight_layout()
plt.savefig('dataset_eda_plot.png', dpi=150)
plt.show()

## Step 5: Feature Matrix & Stratified Train-Test Split

In [ ]:
X = df[['cpu_demand', 'ram_demand', 'storage_demand', 'active_tasks', 'network_io_mbps']]
y = df['demand_tier']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'Training Samples: {len(X_train)}')
print(f'Testing Samples:  {len(X_test)}')

## Step 6: Model Training & Comparative Benchmark
We evaluate 3 popular classification algorithms:
1. **Random Forest Classifier** (Ensemble bagging)
2. **Decision Tree Classifier** (Single tree)
3. **Logistic Regression** (Linear baseline)

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f'{name:<20} | Test Accuracy: {acc*100:.2f}%')

best_name = max(results, key=results.get)
print(f'\nSelected Best Model: {best_name} with {results[best_name]*100:.2f}% accuracy')

## Step 7: Confusion Matrix & Detailed Metrics

In [ ]:
best_clf = models[best_name]
y_preds = best_clf.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_preds, target_names=['LOW', 'MEDIUM', 'HIGH']))

cm = confusion_matrix(y_test, y_preds)
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix: {best_name}')
plt.colorbar()
tick_marks = np.arange(3)
plt.xticks(tick_marks, ['LOW', 'MEDIUM', 'HIGH'])
plt.yticks(tick_marks, ['LOW', 'MEDIUM', 'HIGH'])
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i, j]), horizontalalignment='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.ylabel('Actual Demand Tier')
plt.xlabel('Predicted Demand Tier')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Step 8: Cloud Workload Feature Importance Analysis

In [ ]:
importances = best_clf.feature_importances_
features = X.columns

plt.figure(figsize=(8, 4))
plt.barh(features, importances, color='teal', edgecolor='black')
plt.title('Feature Importance for Cloud Dynamic Scaling')
plt.xlabel('Importance Weight')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

for f, imp in sorted(zip(features, importances), key=lambda x: x[1], reverse=True):
    print(f'{f:<18}: {imp*100:.2f}%')

## Step 9: Model Export & Real-Time Production Inference Test

In [ ]:
# Export trained Random Forest model
joblib.dump(best_clf, 'cloud_resource_allocator_model.joblib')
print('Production Model saved: cloud_resource_allocator_model.joblib')

# Sample Live Cloud Workload Query
sample_telemetry = pd.DataFrame([{
    'cpu_demand': 88.0,
    'ram_demand': 76.5,
    'storage_demand': 60.0,
    'active_tasks': 95,
    'network_io_mbps': 520.0
}])

pred = best_clf.predict(sample_telemetry)[0]
tier_desc = {
    0: 'LOW - Minimal allocation (1 Micro VM)',
    1: 'MEDIUM - Standard allocation (2 Standard VMs)',
    2: 'HIGH - Urgent Auto-scale Triggered (4 High-Performance VMs + Extra Storage Volume)'
}
print('\nInference Result for Sample Input:')
print('Predicted Tier:', tier_desc[pred])